# Diniz Ch.4 — QAM equalization with APA and NLMS

比較 μ=0.1、0.4、0.8，APA 使用 P=2（共 3 個 regressor）。SNR=20 是線性比值，約 13.01 dB。

來源：`Diniz-Chap4-Eq-prob.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Parameters and shared data

In [ ]:
from adsp.experiments.diniz import QAM_CHANNEL

n_samples, n_trials, seed = 5000, 1, 10
n_taps, decision_delay, snr_linear = 10, 5, 20.0
make_data = partial(data.equalization, n_samples, h=QAM_CHANNEL, decision_delay=decision_delay, symbol_kind="qam4",
                    symbol_energy=2.0, snr_linear=snr_linear)

## 3. Model factories and training

In [ ]:
models = {}
for mu in (0.1, 0.4, 0.8):
  models[f"APA μ={mu}"] = partial(af.AffineProjection, filter_order=n_taps - 1, mu=mu, P=2, delta=1e-3)
  models[f"NLMS μ={mu}"] = partial(af.NLMS, filter_order=n_taps - 1, mu=mu, tau=1e-3)
results = training.compare(models, make_data, n_trials=n_trials, seed=seed)
w_reference = ae.makeEqW0(QAM_CHANNEL, n_taps, decision_delay)[0]

## 4. Learning curves, constellations and coefficients

In [ ]:
plot.learning(results, window=64, title="QAM: APA and NLMS")
plot.show()
apa, nlms = results["APA μ=0.4"].representative, results["NLMS μ=0.4"].representative
plot.constellation({"Received": nlms.x, "APA": apa.y, "NLMS": nlms.y}, start=1000)
plot.show()
fig, grid = plot.axes(rows=2, cols=2)
for row, result in enumerate((apa, nlms)):
  plot.lines({
    "Real": result.w_final.real,
    "Imaginary": result.w_final.imag,
    "Reference real": w_reference.real
  }, ax=grid[row, 0], xlabel="Tap index", title=f"{result.name}: stored weights")
  cascade = np.convolve(QAM_CHANNEL, np.conj(result.w_final))
  plot.stem({"Cascade magnitude": np.abs(cascade)}, ax=grid[row, 1], title=f"{result.name}: cascade")
plot.show(fig)
training.summary(results)